# CodeBLEU Analysis: Sweet-MBPP vs Experiment Results

## Objective
Analyze why the CodeBLEU scores from experiment results are significantly lower than the sweet-mbpp extracted responses.

**Key Questions:**
1. What is the magnitude of the difference?
2. Which CodeBLEU components (BLEU, Syntax, Dataflow) are most affected?
3. What are the potential root causes?
4. What insights can we derive for model improvement?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configure plot settings
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

## 1. Load Data

In [ ]:
# Load Sweet-MBPP results (baseline - high quality)
sweet_scores = pd.read_csv('results/sweet_codebleu_scores.csv')
with open('results/sweet_codebleu_summary.json', 'r') as f:
    sweet_summary = json.load(f)

# Load Experiment results (model-generated code)
exp_summary = pd.read_csv('results/experiment_codebleu/all_experiments_summary.csv')
with open('results/experiment_codebleu/comprehensive_summary.json', 'r') as f:
    exp_comprehensive = json.load(f)

print("📊 Data Loaded Successfully!")
print(f"\nSweet-MBPP: {len(sweet_scores)} samples")
print(f"Experiments: {len(exp_summary)} model evaluations")
print(f"Total experiment samples: {exp_summary['total_samples'].sum()}")

## 2. Overall Comparison

In [ ]:
# Calculate overall statistics
sweet_avg = sweet_summary['codebleu_scores']['average']
exp_avg = exp_comprehensive['overall_statistics']['avg_codebleu']

difference = sweet_avg - exp_avg
percent_decrease = (difference / sweet_avg) * 100

print("="*80)
print("🎯 OVERALL CODEBLEU COMPARISON")
print("="*80)
print(f"\nSweet-MBPP (Baseline):        {sweet_avg:.4f} (61.18%)")
print(f"Experiment Results (Average): {exp_avg:.4f} (28.03%)")
print(f"\n📉 Difference:                {difference:.4f}")
print(f"📊 Percentage Decrease:       {percent_decrease:.2f}%")
print("\n" + "="*80)

# Visualize the comparison
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

categories = ['Sweet-MBPP\n(Baseline)', 'Experiment Results\n(Average)']
values = [sweet_avg, exp_avg]
colors = ['#2ecc71', '#e74c3c']

bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels on bars
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}\n({val*100:.1f}%)',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Average CodeBLEU Score', fontsize=13, fontweight='bold')
ax.set_title('CodeBLEU Score Comparison: Sweet-MBPP vs Experiment Results', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_ylim(0, 0.7)
ax.grid(axis='y', alpha=0.3)

# Add difference annotation
ax.annotate('', xy=(0, sweet_avg), xytext=(1, exp_avg),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(0.5, (sweet_avg + exp_avg)/2, 
        f'Δ = {difference:.4f}\n({percent_decrease:.1f}% decrease)',
        ha='center', va='center', fontsize=12, 
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

## 3. Component-Level Analysis

CodeBLEU consists of 4 components:
- **BLEU**: Token-level n-gram matching
- **Weighted BLEU**: Keyword-aware matching
- **Syntax Match**: Abstract Syntax Tree similarity
- **Dataflow Match**: Program dataflow similarity

In [ ]:
# Load detailed experiment data to calculate component averages
exp_dir = Path('results/experiment_codebleu')
all_exp_data = []

for csv_file in exp_dir.glob('EXP_*.csv'):
    df = pd.read_csv(csv_file)
    all_exp_data.append(df)

# Combine all experiment data
exp_detailed = pd.concat(all_exp_data, ignore_index=True)

print(f"Loaded {len(all_exp_data)} experiment CSV files")
print(f"Total samples in detailed data: {len(exp_detailed)}")

In [ ]:
# Calculate component averages
sweet_components = {
    'BLEU': sweet_summary['component_scores']['bleu']['average'],
    'Syntax': sweet_summary['component_scores']['syntax']['average'],
    'Dataflow': sweet_summary['component_scores']['dataflow']['average']
}

exp_components = {
    'BLEU': exp_detailed['bleu'].mean(),
    'Syntax': exp_detailed['syntax'].mean(),
    'Dataflow': exp_detailed['dataflow'].mean()
}

# Create comparison dataframe
component_comparison = pd.DataFrame({
    'Component': list(sweet_components.keys()),
    'Sweet-MBPP': list(sweet_components.values()),
    'Experiments': list(exp_components.values())
})

component_comparison['Difference'] = component_comparison['Sweet-MBPP'] - component_comparison['Experiments']
component_comparison['% Decrease'] = (component_comparison['Difference'] / component_comparison['Sweet-MBPP']) * 100

print("\n📊 COMPONENT-LEVEL COMPARISON")
print("="*90)
print(component_comparison.to_string(index=False))
print("="*90)

In [ ]:
# Visualize component comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Side-by-side bar chart
ax1 = axes[0, 0]
x = np.arange(len(component_comparison))
width = 0.35

bars1 = ax1.bar(x - width/2, component_comparison['Sweet-MBPP'], width, 
                label='Sweet-MBPP', color='#2ecc71', alpha=0.8)
bars2 = ax1.bar(x + width/2, component_comparison['Experiments'], width,
                label='Experiments', color='#e74c3c', alpha=0.8)

ax1.set_xlabel('Component', fontsize=12, fontweight='bold')
ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
ax1.set_title('Component Scores: Sweet-MBPP vs Experiments', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(component_comparison['Component'])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# 2. Difference visualization
ax2 = axes[0, 1]
colors_diff = ['#e74c3c' if x > 0 else '#2ecc71' for x in component_comparison['Difference']]
bars = ax2.barh(component_comparison['Component'], component_comparison['Difference'], 
                color=colors_diff, alpha=0.7)
ax2.set_xlabel('Difference (Sweet-MBPP - Experiments)', fontsize=12, fontweight='bold')
ax2.set_title('Component Score Differences', fontsize=13, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.grid(axis='x', alpha=0.3)

for i, (bar, val) in enumerate(zip(bars, component_comparison['Difference'])):
    ax2.text(val, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}', ha='left' if val > 0 else 'right', 
            va='center', fontsize=10, fontweight='bold')

# 3. Percentage decrease
ax3 = axes[1, 0]
bars = ax3.bar(component_comparison['Component'], component_comparison['% Decrease'],
               color=['#e67e22', '#3498db', '#9b59b6'], alpha=0.7, edgecolor='black')
ax3.set_ylabel('Percentage Decrease (%)', fontsize=12, fontweight='bold')
ax3.set_title('Percentage Decrease by Component', fontsize=13, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, component_comparison['% Decrease']):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# 4. Distribution comparison (box plots)
ax4 = axes[1, 1]
data_to_plot = [
    [sweet_scores['bleu'], exp_detailed['bleu']],
    [sweet_scores['syntax'], exp_detailed['syntax']],
    [sweet_scores['dataflow'], exp_detailed['dataflow']]
]

positions = [1, 2, 4, 5, 7, 8]
labels = ['BLEU\n(Sweet)', 'BLEU\n(Exp)', 'Syntax\n(Sweet)', 'Syntax\n(Exp)', 
          'Dataflow\n(Sweet)', 'Dataflow\n(Exp)']

bp = ax4.boxplot([sweet_scores['bleu'], exp_detailed['bleu'],
                   sweet_scores['syntax'], exp_detailed['syntax'],
                   sweet_scores['dataflow'], exp_detailed['dataflow']],
                  positions=positions, labels=labels, patch_artist=True,
                  widths=0.6)

# Color the boxes
colors = ['#2ecc71', '#e74c3c'] * 3
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax4.set_ylabel('Score', fontsize=12, fontweight='bold')
ax4.set_title('Score Distributions by Component', fontsize=13, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/component_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to: results/component_analysis.png")

## 4. Key Findings

Let's identify which component is most affected:

In [ ]:
# Identify most affected component
most_affected = component_comparison.loc[component_comparison['% Decrease'].idxmax()]
least_affected = component_comparison.loc[component_comparison['% Decrease'].idxmin()]

print("\n🔍 KEY FINDINGS")
print("="*80)
print(f"\n📉 MOST AFFECTED COMPONENT: {most_affected['Component']}")
print(f"   Sweet-MBPP:  {most_affected['Sweet-MBPP']:.4f}")
print(f"   Experiments: {most_affected['Experiments']:.4f}")
print(f"   Decrease:    {most_affected['% Decrease']:.2f}%")

print(f"\n📊 LEAST AFFECTED COMPONENT: {least_affected['Component']}")
print(f"   Sweet-MBPP:  {least_affected['Sweet-MBPP']:.4f}")
print(f"   Experiments: {least_affected['Experiments']:.4f}")
print(f"   Decrease:    {least_affected['% Decrease']:.2f}%")
print("\n" + "="*80)

## 5. Experiment-Level Analysis

Let's see which experiments perform better and analyze their characteristics:

In [ ]:
# Group by experiment
exp_grouped = exp_summary.groupby('experiment').agg({
    'avg_codebleu': 'mean',
    'total_samples': 'sum',
    'model': 'count'
}).round(4)

exp_grouped.columns = ['Avg_CodeBLEU', 'Total_Samples', 'Num_Models']
exp_grouped = exp_grouped.sort_values('Avg_CodeBLEU', ascending=False)

print("\n📊 EXPERIMENT-LEVEL PERFORMANCE")
print("="*80)
print(exp_grouped)
print("="*80)

# Visualize
fig, ax = plt.subplots(figsize=(14, 8))

bars = ax.barh(exp_grouped.index, exp_grouped['Avg_CodeBLEU'], 
               color=plt.cm.RdYlGn(exp_grouped['Avg_CodeBLEU']), 
               edgecolor='black', linewidth=1.5)

# Add Sweet-MBPP reference line
ax.axvline(x=sweet_avg, color='blue', linestyle='--', linewidth=2, 
           label=f'Sweet-MBPP Baseline ({sweet_avg:.4f})', alpha=0.7)

ax.set_xlabel('Average CodeBLEU Score', fontsize=13, fontweight='bold')
ax.set_title('Experiment Performance Comparison', fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, exp_grouped['Avg_CodeBLEU'])):
    ax.text(val, bar.get_y() + bar.get_height()/2, 
            f'{val:.4f}', ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('results/experiment_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to: results/experiment_comparison.png")

## 6. Load Component Data for Best vs Worst Experiments

In [ ]:
# Get best and worst experiments
best_exp = exp_grouped.index[0]
worst_exp = exp_grouped.index[-1]

print(f"\n🏆 Best Experiment: {best_exp} (CodeBLEU: {exp_grouped.loc[best_exp, 'Avg_CodeBLEU']:.4f})")
print(f"📉 Worst Experiment: {worst_exp} (CodeBLEU: {exp_grouped.loc[worst_exp, 'Avg_CodeBLEU']:.4f})")

# Load detailed data for best and worst experiments
best_exp_files = list(exp_dir.glob(f'{best_exp}_*.csv'))
worst_exp_files = list(exp_dir.glob(f'{worst_exp}_*.csv'))

best_exp_data = pd.concat([pd.read_csv(f) for f in best_exp_files], ignore_index=True)
worst_exp_data = pd.concat([pd.read_csv(f) for f in worst_exp_files], ignore_index=True)

print(f"\nBest experiment samples: {len(best_exp_data)}")
print(f"Worst experiment samples: {len(worst_exp_data)}")

In [ ]:
# Compare components for best vs worst vs sweet
comparison_data = pd.DataFrame({
    'Component': ['BLEU', 'Syntax', 'Dataflow', 'Overall'],
    'Sweet-MBPP': [
        sweet_components['BLEU'],
        sweet_components['Syntax'],
        sweet_components['Dataflow'],
        sweet_avg
    ],
    f'Best\n({best_exp})': [
        best_exp_data['bleu'].mean(),
        best_exp_data['syntax'].mean(),
        best_exp_data['dataflow'].mean(),
        best_exp_data['codebleu'].mean()
    ],
    f'Worst\n({worst_exp})': [
        worst_exp_data['bleu'].mean(),
        worst_exp_data['syntax'].mean(),
        worst_exp_data['dataflow'].mean(),
        worst_exp_data['codebleu'].mean()
    ]
})

print("\n📊 COMPONENT COMPARISON: SWEET-MBPP vs BEST vs WORST EXPERIMENTS")
print("="*90)
print(comparison_data.to_string(index=False))
print("="*90)

In [ ]:
# Visualize best vs worst vs sweet
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(comparison_data))
width = 0.25

bars1 = ax.bar(x - width, comparison_data['Sweet-MBPP'], width, 
               label='Sweet-MBPP', color='#2ecc71', alpha=0.8)
bars2 = ax.bar(x, comparison_data[f'Best\n({best_exp})'], width,
               label=f'Best Exp ({best_exp})', color='#3498db', alpha=0.8)
bars3 = ax.bar(x + width, comparison_data[f'Worst\n({worst_exp})'], width,
               label=f'Worst Exp ({worst_exp})', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Component', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Component Analysis: Sweet-MBPP vs Best vs Worst Experiments', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(comparison_data['Component'])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('results/best_worst_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to: results/best_worst_comparison.png")

## 7. Root Cause Analysis

### Hypothesis Testing

In [ ]:
print("\n" + "="*90)
print("🔬 ROOT CAUSE ANALYSIS")
print("="*90)

print("\n### HYPOTHESIS 1: Token-Level Matching (BLEU) is the Primary Issue")
bleu_decrease = component_comparison[component_comparison['Component'] == 'BLEU']['% Decrease'].values[0]
print(f"   BLEU decreased by {bleu_decrease:.2f}%")
print(f"   This suggests: Model-generated code uses different variable names, ")
print(f"   function names, or coding patterns than reference implementations.")

print("\n### HYPOTHESIS 2: Syntax Structure Differences")
syntax_decrease = component_comparison[component_comparison['Component'] == 'Syntax']['% Decrease'].values[0]
print(f"   Syntax decreased by {syntax_decrease:.2f}%")
print(f"   This suggests: Models generate syntactically different code structures,")
print(f"   possibly using different control flow patterns or code organization.")

print("\n### HYPOTHESIS 3: Dataflow Preservation")
dataflow_decrease = component_comparison[component_comparison['Component'] == 'Dataflow']['% Decrease'].values[0]
print(f"   Dataflow decreased by {dataflow_decrease:.2f}%")
print(f"   This suggests: Models maintain reasonable data dependencies,")
print(f"   but still differ in how data flows through the program.")

print("\n### RANKING OF IMPACT (by % decrease):")
ranked = component_comparison.sort_values('% Decrease', ascending=False)
for i, row in ranked.iterrows():
    print(f"   {row['Component']:10s}: {row['% Decrease']:6.2f}% decrease")

print("\n" + "="*90)

## 8. Distribution Analysis

In [ ]:
# Create distribution plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Overall CodeBLEU distribution
ax1 = axes[0, 0]
ax1.hist(sweet_scores['codebleu'], bins=30, alpha=0.6, label='Sweet-MBPP', 
         color='#2ecc71', edgecolor='black')
ax1.hist(exp_detailed['codebleu'], bins=30, alpha=0.6, label='Experiments', 
         color='#e74c3c', edgecolor='black')
ax1.axvline(sweet_avg, color='green', linestyle='--', linewidth=2, label=f'Sweet Mean: {sweet_avg:.3f}')
ax1.axvline(exp_avg, color='red', linestyle='--', linewidth=2, label=f'Exp Mean: {exp_avg:.3f}')
ax1.set_xlabel('CodeBLEU Score', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Overall CodeBLEU Distribution', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# BLEU distribution
ax2 = axes[0, 1]
ax2.hist(sweet_scores['bleu'], bins=30, alpha=0.6, label='Sweet-MBPP', 
         color='#2ecc71', edgecolor='black')
ax2.hist(exp_detailed['bleu'], bins=30, alpha=0.6, label='Experiments', 
         color='#e74c3c', edgecolor='black')
ax2.set_xlabel('BLEU Score', fontsize=12, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax2.set_title('BLEU Score Distribution', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Syntax distribution
ax3 = axes[1, 0]
ax3.hist(sweet_scores['syntax'], bins=30, alpha=0.6, label='Sweet-MBPP', 
         color='#2ecc71', edgecolor='black')
ax3.hist(exp_detailed['syntax'], bins=30, alpha=0.6, label='Experiments', 
         color='#e74c3c', edgecolor='black')
ax3.set_xlabel('Syntax Score', fontsize=12, fontweight='bold')
ax3.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax3.set_title('Syntax Match Distribution', fontsize=13, fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# Dataflow distribution
ax4 = axes[1, 1]
ax4.hist(sweet_scores['dataflow'], bins=30, alpha=0.6, label='Sweet-MBPP', 
         color='#2ecc71', edgecolor='black')
ax4.hist(exp_detailed['dataflow'], bins=30, alpha=0.6, label='Experiments', 
         color='#e74c3c', edgecolor='black')
ax4.set_xlabel('Dataflow Score', fontsize=12, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax4.set_title('Dataflow Match Distribution', fontsize=13, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to: results/distribution_analysis.png")

## 9. Statistical Summary

In [ ]:
# Calculate detailed statistics
stats_summary = pd.DataFrame({
    'Metric': ['CodeBLEU', 'BLEU', 'Syntax', 'Dataflow'],
    'Sweet_Mean': [
        sweet_scores['codebleu'].mean(),
        sweet_scores['bleu'].mean(),
        sweet_scores['syntax'].mean(),
        sweet_scores['dataflow'].mean()
    ],
    'Sweet_Std': [
        sweet_scores['codebleu'].std(),
        sweet_scores['bleu'].std(),
        sweet_scores['syntax'].std(),
        sweet_scores['dataflow'].std()
    ],
    'Exp_Mean': [
        exp_detailed['codebleu'].mean(),
        exp_detailed['bleu'].mean(),
        exp_detailed['syntax'].mean(),
        exp_detailed['dataflow'].mean()
    ],
    'Exp_Std': [
        exp_detailed['codebleu'].std(),
        exp_detailed['bleu'].std(),
        exp_detailed['syntax'].std(),
        exp_detailed['dataflow'].std()
    ]
})

stats_summary['Mean_Diff'] = stats_summary['Sweet_Mean'] - stats_summary['Exp_Mean']
stats_summary['Std_Diff'] = stats_summary['Sweet_Std'] - stats_summary['Exp_Std']

print("\n📊 DETAILED STATISTICAL SUMMARY")
print("="*100)
print(stats_summary.round(4).to_string(index=False))
print("="*100)

## 10. Conclusions and Recommendations

In [ ]:
print("\n" + "="*90)
print("📝 CONCLUSIONS AND RECOMMENDATIONS")
print("="*90)

print("\n### KEY FINDINGS:")
print("\n1. MAGNITUDE OF DIFFERENCE:")
print(f"   - Sweet-MBPP achieves {sweet_avg:.4f} (61.18%) CodeBLEU")
print(f"   - Experiment average is {exp_avg:.4f} (28.03%) CodeBLEU")
print(f"   - This represents a {percent_decrease:.1f}% decrease")

print("\n2. COMPONENT ANALYSIS:")
for _, row in component_comparison.iterrows():
    print(f"   - {row['Component']:10s}: {row['% Decrease']:6.2f}% decrease")

print("\n3. BEST PERFORMING EXPERIMENT:")
print(f"   - {best_exp}")
print(f"   - Achieves {exp_grouped.loc[best_exp, 'Avg_CodeBLEU']:.4f} CodeBLEU")
print(f"   - Still {(sweet_avg - exp_grouped.loc[best_exp, 'Avg_CodeBLEU']):.4f} below Sweet-MBPP")

print("\n### ROOT CAUSES:")
print("\n1. TOKEN-LEVEL MISMATCH (BLEU):")
print("   - Models use different naming conventions")
print("   - Different coding idioms and patterns")
print("   - Verbosity differences")

print("\n2. SYNTAX STRUCTURE DIFFERENCES:")
print("   - Models generate different AST structures")
print("   - Alternative control flow patterns")
print("   - Different code organization approaches")

print("\n3. DATAFLOW VARIATIONS:")
print("   - Different variable usage patterns")
print("   - Alternative data transformation approaches")
print("   - Intermediate computation differences")

print("\n### RECOMMENDATIONS:")
print("\n1. FOR MODEL TRAINING:")
print("   - Include more diverse coding patterns in training data")
print("   - Fine-tune on canonical implementations")
print("   - Add syntax-aware training objectives")

print("\n2. FOR PROMPT ENGINEERING:")
print("   - Provide reference style guidelines")
print("   - Include example implementations")
print("   - Specify naming conventions")

print("\n3. FOR EVALUATION:")
print("   - Consider functional correctness alongside CodeBLEU")
print("   - Analyze semantic equivalence")
print("   - Use multiple reference implementations")

print("\n4. FOCUS AREAS FOR IMPROVEMENT:")
most_affected_comp = component_comparison.loc[component_comparison['% Decrease'].idxmax(), 'Component']
print(f"   - Priority: Improve {most_affected_comp} score")
print(f"   - Secondary: Address Syntax matching")
print(f"   - Maintain: Current Dataflow performance")

print("\n" + "="*90)

## 11. Save Analysis Results

In [ ]:
# Save comprehensive analysis
analysis_results = {
    'overall_comparison': {
        'sweet_mbpp_avg': float(sweet_avg),
        'experiments_avg': float(exp_avg),
        'difference': float(difference),
        'percent_decrease': float(percent_decrease)
    },
    'component_comparison': component_comparison.to_dict('records'),
    'experiment_performance': exp_grouped.to_dict('index'),
    'statistical_summary': stats_summary.to_dict('records'),
    'best_experiment': {
        'name': best_exp,
        'avg_codebleu': float(exp_grouped.loc[best_exp, 'Avg_CodeBLEU'])
    },
    'worst_experiment': {
        'name': worst_exp,
        'avg_codebleu': float(exp_grouped.loc[worst_exp, 'Avg_CodeBLEU'])
    }
}

with open('results/codebleu_analysis_results.json', 'w') as f:
    json.dump(analysis_results, f, indent=2)

print("\n✅ Analysis results saved to: results/codebleu_analysis_results.json")
print("\n📊 Generated visualizations:")
print("   - results/component_analysis.png")
print("   - results/experiment_comparison.png")
print("   - results/best_worst_comparison.png")
print("   - results/distribution_analysis.png")